# V3 · Thời gian và thời tiết

**Mục tiêu:** trả lời câu hỏi nghiên cứu 2: thêm thời tiết cải thiện dự đoán bao nhiêu so với cấu hình A. So sánh A–B dùng cùng mô hình, cùng lưới tham số, cùng mốc chia và cùng 6.086 giờ validation.

**Kịch bản:** dự đoán lưu lượng của một giờ khi đã biết thời gian và thời tiết **quan trắc** của giờ đó. Kết quả không đồng nghĩa với khả năng dự đoán khi chỉ có dự báo thời tiết.

**Chạy trước:** `V_2.ipynb`, vì notebook này đọc dự đoán của cấu hình A từ `experiments/experiments_logs.xlsx`. Test giữ riêng.

**Tuning theo trang 4.** Notebook hiển thị mốc fold và bảng MAE từng ứng viên. `src/tuning.py` thực hiện chia fold và tìm tham số. Với cấu hình thời tiết, mỗi fold dùng quan sát gốc để học median từ phần thời gian trước validation. LR là mốc không tuning. RF, GB và MLP theo sơ đồ, Ridge và XGBoost giữ để đối chứng. Kết quả đã lưu trong notebook là kết quả lịch sử. Khi chạy lại V2 → V3 → V4, kết quả mới lưu trong `experiments/experiments_logs.xlsx`.

## 3.1 · Chuẩn bị dữ liệu

In [2]:
from pathlib import Path
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost

# Tìm gốc dự án từ thư mục hiện tại hoặc các thư mục cha.
current_dir = Path.cwd().resolve()
project_dir = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "data" / "Metro_Interstate_Traffic_Volume.csv").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Không tìm thấy CSV trong thư mục dự án.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

output_dir = project_dir / "experiments"
figure_dir = project_dir / "figures"
pd.set_option("display.max_columns", None)
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e4e0", "axes.axisbelow": True,
    "lines.linewidth": 2,
})
BLUE, ORANGE = "#2a78d6", "#eb6834"

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

Python: 3.14.7
pandas: 3.0.6 | scikit-learn: 1.9.1 | xgboost: 3.4.1


In [3]:
from src.data import DATA_PATH as data_path
from src.data import build_hourly_table, fill_missing_weather, load_raw, load_splits, split_chronological
from src.evaluation import bootstrap_mae_difference, regression_metrics, run_observations, weekday_hour_baseline
from src.experiment_logger import save_experiment_log
from src.features import CONFIG_B, WEATHER_GROUP_FEATURES, WEATHER_GROUPS, build_features
from src.tuning import describe_folds, run_config, summarize_search
from src.models import MODEL_NAMES, make_model, results_table
from src.data_validation import validate_structure_and_time
from src.experiment_excel import save_experiment_workbook, load_experiment_workbook, validate_prediction_rows

### 3.1.1 · Đọc và kiểm tra CSV gốc

In [4]:
raw = load_raw(data_path)
quality = validate_structure_and_time(raw)
display(pd.Series(quality, name="Kiểm tra CSV"))
display(raw.head(3))

row_count                                    48204
column_count                                     9
unique_timestamp_count                       40575
conflicting_timestamp_count                      0
start_time                     2012-10-02 09:00:00
end_time                       2018-09-30 23:00:00
is_time_sorted                                True
Name: Kiểm tra CSV, dtype: object

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767


### 3.1.2 · Gộp theo giờ và chia 70/15/15

In [5]:
hourly = build_hourly_table(raw)
assert len(hourly) == quality["unique_timestamp_count"]
raw_splits = split_chronological(hourly)
assert tuple(map(len, raw_splits.values())) == (28402, 6086, 6087)
assert raw_splits["train"]["date_time"].max() < raw_splits["validation"]["date_time"].min()
assert raw_splits["validation"]["date_time"].max() < raw_splits["test"]["date_time"].min()
display(pd.DataFrame({
    name: {"Số giờ": len(part), "Bắt đầu": part["date_time"].min(), "Kết thúc": part["date_time"].max()}
    for name, part in raw_splits.items()
}).T)

,Số giờ,Bắt đầu,Kết thúc
train,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
validation,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
test,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


### 3.1.3 · Chuẩn bị đặc trưng cho train và validation

Giữ bảng theo giờ chưa điền để CV học lại median từ dữ liệu gốc trong mỗi fold.

In [6]:
model_splits = {name: raw_splits[name] for name in ("train", "validation")}
splits = {**fill_missing_weather(model_splits), "test": raw_splits["test"]}
features = build_features({name: splits[name] for name in model_splits})
train, validation = features["train"], features["validation"]
for name, frame in features.items():
    pd.testing.assert_frame_equal(frame[["date_time", "traffic_volume"]].reset_index(drop=True),
                                  raw_splits[name][["date_time", "traffic_volume"]].reset_index(drop=True))
display(pd.DataFrame({
    name: raw_splits[name][["temp", "rain_1h"]].isna().sum()
    for name in model_splits
}).T.rename(columns={"temp": "Thiếu nhiệt độ trước khi điền", "rain_1h": "Thiếu mưa trước khi điền"}))

,Thiếu nhiệt độ trước khi điền,Thiếu mưa trước khi điền
train,10,1
validation,0,0


## 3.2 · Biểu diễn nhãn thời tiết

Sau khi gộp theo giờ, `weather_main` có thể chứa nhiều nhãn như `drizzle | rain`. Phần này kiểm tra lựa chọn giữa one-hot trên chuỗi gộp và cờ 0/1 theo từng nhóm.

In [7]:
combos_train = set(train["weather_main"])
combos_validation = set(validation["weather_main"])
print("Tỷ lệ giờ train có nhiều nhãn:", f"{train['weather_main'].str.contains(' | ', regex=False).mean():.1%}")
print("Số tổ hợp nhãn trong train:", len(combos_train))
print("Số tổ hợp trong validation:", len(combos_validation))
print("Tổ hợp chỉ có trong validation:", sorted(combos_validation - combos_train))

display(pd.DataFrame({
    "train": train[WEATHER_GROUP_FEATURES].sum(),
    "validation": validation[WEATHER_GROUP_FEATURES].sum(),
}).sort_values("train", ascending=False))
assert train[WEATHER_GROUP_FEATURES].sum(axis=1).ge(1).all()

Tỷ lệ giờ train có nhiều nhãn: 12.4%
Số tổ hợp nhãn trong train: 76
Số tổ hợp trong validation: 45
Tổ hợp chỉ có trong validation: ['fog | rain | thunderstorm', 'haze | thunderstorm']


,train,validation
weather_clouds,11129,2120
weather_clear,9135,2115
weather_mist,4011,905
weather_rain,3788,783
weather_snow,1989,362
weather_drizzle,1161,363
weather_haze,895,228
weather_fog,567,162
weather_thunderstorm,507,235
weather_smoke,17,1


**Nhận xét**

- 12,4% số giờ train mang nhiều nhãn `weather_main`. Train có 76 tổ hợp nhãn, validation có 45, trong đó `fog | rain | thunderstorm` và `haze | thunderstorm` chỉ xuất hiện ở validation.
- One-hot trên chuỗi gộp sẽ bỏ qua hai tổ hợp này và tách các tổ hợp hiếm thành cột riêng. Cờ theo 11 nhóm giữ được thông tin của từng nhãn và không gặp nhãn lạ.
- `smoke` (17 giờ train) và `squall` (4 giờ train, 0 giờ validation) quá hiếm để mô hình học ổn định. `snow_1h > 0` chỉ có 31 giờ train và 0 giờ validation; thông tin tuyết chủ yếu nằm ở nhãn `snow`.

## 3.3 · Thí nghiệm và tuning cấu hình B

### 3.3.1 · Kiểm tra đầu vào mô hình và ranh giới CV

**Kiểm tra kết quả đối chứng trước khi tuning.** Chạy và lưu V2 vào `experiments_logs.xlsx` trước. Cell này kiểm tra đúng CSV, đúng các giờ validation và đủ dự đoán cho mọi mô hình.

In [8]:
try:
    _, previous, params_a_all = load_experiment_workbook("V_2", output_dir, data_path)
except FileNotFoundError as error:
    raise FileNotFoundError(
        "Hãy chạy V_2.ipynb đến hết cell lưu kết quả, sau đó chạy lại cell này."
    ) from error
validate_prediction_rows(previous, validation)
missing_models = [name for name in MODEL_NAMES if f"A_lich_le|{name}" not in previous]
if missing_models:
    raise ValueError(f"Workbook V_2 thiếu dự đoán của {', '.join(missing_models)}. Hãy chạy và lưu lại V_2.")
print("Đã kiểm tra kết quả đối chứng:", previous.attrs["run_id"])

Đã kiểm tra kết quả đối chứng: V2_20261107_163034_1e5e45c4


In [9]:
configurations = {"B_thoi_tiet": CONFIG_B}
checks = [(configurations, train, validation)]
display(describe_folds(train))

,fold,fit_start,fit_end,validation_start,validation_end,n_fit,n_validation
0,1,2012-10-02 09:00:00,2013-08-18 18:00:00,2013-08-18 19:00:00,2015-07-07 11:00:00,7102,7100
1,2,2012-10-02 09:00:00,2015-07-07 11:00:00,2015-07-07 12:00:00,2016-07-15 04:00:00,14202,7100
2,3,2012-10-02 09:00:00,2016-07-15 04:00:00,2016-07-15 05:00:00,2017-05-10 04:00:00,21302,7100


In [10]:
for configs, fit_frame, score_frame in checks:
    for name, columns in configs.items():
        assert len(columns) == len(set(columns)) and "traffic_volume" not in columns
        assert set(columns) <= set(fit_frame) & set(score_frame)
        assert all(pd.api.types.is_numeric_dtype(fit_frame[col]) for col in columns)
        assert all(pd.api.types.is_numeric_dtype(score_frame[col]) for col in columns)
        assert not np.isinf(fit_frame[columns].to_numpy(dtype=float)).any()
        assert np.isfinite(score_frame[columns].to_numpy(dtype=float)).all()
        allowed_missing = set(HISTORY_FEATURES) if name == "C_toan_bo_train" else set()
        assert set(fit_frame[columns].columns[fit_frame[columns].isna().any()]) <= allowed_missing
        print(f"{name}: {len(fit_frame):,} giờ train, {len(score_frame):,} giờ validation, {len(columns)} cột")

B_thoi_tiet: 28,402 giờ train, 6,086 giờ validation, 27 cột


### 3.3.2 · Thí nghiệm và tuning

LR chạy một cấu hình làm mốc. Các mô hình còn lại chọn tham số bằng MAE trung bình trên các fold của train.

In [11]:
results = run_config("B_thoi_tiet", CONFIG_B, train, validation, raw_train=raw_splits["train"])

e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMa

### 3.3.3 · Đọc tham số đã chọn và MAE từng fold

In [12]:
for item in results:
    print(f"{item['config']} | {item['model']} | {item['search'].best_params_}")
    display(summarize_search(item["search"]).head(3).round(3))

B_thoi_tiet | Linear Regression | {}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,{},569.132,572.518,577.075,572.908,1


B_thoi_tiet | Random Forest | {'min_samples_leaf': 5, 'max_features': 0.33, 'max_depth': 20}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'min_samples_leaf': 5, 'max_features': 0.33, ...",262.468,266.329,306.174,278.323,1
1,"{'min_samples_leaf': 10, 'max_features': 0.5, ...",263.687,264.244,308.474,278.802,2
2,"{'min_samples_leaf': 2, 'max_features': 0.33, ...",263.413,267.251,310.573,280.412,3


B_thoi_tiet | Gradient Boosting | {'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.1}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'model__n_estimators': 200, 'model__max_depth...",294.733,287.224,333.506,305.154,1
1,"{'model__n_estimators': 200, 'model__max_depth...",340.651,350.383,387.955,359.663,2
2,"{'model__n_estimators': 100, 'model__max_depth...",395.804,378.166,395.620,389.863,3


... (2 outputs left out)

B_thoi_tiet | Ridge | {'model__alpha': 0.1}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,{'model__alpha': 0.1},290.680,298.613,329.911,306.401,1
1,{'model__alpha': 1.0},299.724,301.572,330.662,310.653,2
2,{'model__alpha': 10.0},471.761,367.972,356.841,398.858,3


B_thoi_tiet | XGBoost | {'subsample': 0.7, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.8}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'subsample': 0.7, 'min_child_weight': 10, 'ma...",274.651,266.609,308.289,283.183,1
1,"{'subsample': 0.7, 'min_child_weight': 1, 'max...",275.397,267.128,308.850,283.792,2
2,"{'subsample': 0.7, 'min_child_weight': 1, 'max...",272.995,267.587,311.739,284.107,3


### 3.3.4 · Đánh giá trên validation và lưu kết quả

In [13]:
display(results_table(results).round(3))

,config,model,so_dac_trung,MAE,RMSE,R2,mean_cv_mae
0,B_thoi_tiet,Linear Regression,27,569.610,736.938,0.859,572.908
1,B_thoi_tiet,Random Forest,27,245.829,399.503,0.959,278.323
2,B_thoi_tiet,Gradient Boosting,27,266.973,416.356,0.955,305.154
3,B_thoi_tiet,MLP,27,269.465,417.357,0.955,319.468
4,B_thoi_tiet,Ridge,27,280.606,450.041,0.948,306.401
5,B_thoi_tiet,XGBoost,27,239.187,384.128,0.962,283.183


## 3.4 · Đối chứng A–B

Chênh lệch dương nghĩa là B có MAE thấp hơn A. Khoảng tin cậy 95% lấy mẫu lại theo ngày trên validation.

In [17]:
rows = []
for item in results:
    prediction_a = previous[f"A_lich_le|{item['model']}"].to_numpy()
    metrics_a = regression_metrics(validation["traffic_volume"], prediction_a)
    gap = bootstrap_mae_difference(validation, prediction_a, item["prediction"])
    rows.append({
        "model": item["model"],
        "MAE_A": metrics_a["MAE"], "MAE_B": item["metrics"]["MAE"],
        "RMSE_A": metrics_a["RMSE"], "RMSE_B": item["metrics"]["RMSE"],
        "R2_A": metrics_a["R2"], "R2_B": item["metrics"]["R2"],
        **gap,
    })
ablation = pd.DataFrame(rows)
ablation["giam_MAE_%"] = 100 * ablation["chenh_lech_MAE"] / ablation["MAE_A"]
display(ablation.round(3))

baseline_prediction = weekday_hour_baseline(train, validation)
baseline_metrics = regression_metrics(validation["traffic_volume"], baseline_prediction)
from importlib import reload
import src.experiment_excel as excel_io

workbook_path = reload(excel_io).save_experiment_workbook(
    version="V_3", results=results, target=validation, splits=splits,
    folds={"B_thoi_tiet": describe_folds(train)}, data_path=data_path, output_dir=output_dir,
    baseline_metrics=baseline_metrics, baseline_predictions=baseline_prediction,
    comparisons={"Đối chứng A-B": ablation}, prior_run=previous.attrs["run_id"],
)
print("Đã lưu:", workbook_path)

,model,MAE_A,MAE_B,RMSE_A,RMSE_B,R2_A,R2_B,chenh_lech_MAE,CI95_thap,CI95_cao,so_ngay,giam_MAE_%
0,Linear Regression,570.570,569.610,739.066,736.938,0.859,0.859,0.960,-1.603,3.619,255,0.168
1,Random Forest,248.705,245.829,396.713,399.503,0.959,0.959,2.876,-3.352,8.574,255,1.156
2,Gradient Boosting,264.792,266.973,412.645,416.356,0.956,0.955,-2.181,-7.296,3.011,255,-0.824
3,MLP,247.978,269.465,386.036,417.357,0.961,0.955,-21.488,-32.790,-11.813,255,-8.665
4,Ridge,283.098,280.606,453.231,450.041,0.947,0.948,2.492,-0.497,5.591,255,0.880
5,XGBoost,240.602,239.187,376.404,384.128,0.963,0.962,1.414,-6.625,9.021,255,0.588


Đã lưu: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\experiments_logs.xlsx


Khoảng tin cậy trên chỉ phản ánh dao động theo ngày của validation, chưa phản ánh dao động do `random_state` của XGBoost (lấy mẫu dòng và cột). Cell sau fit lại hai cấu hình với 5 seed, giữ nguyên tham số đã chọn.

In [18]:
from src.features import CONFIG_A

params_a = params_a_all["A_lich_le|XGBoost"]
params_b = next(item["search"].best_params_ for item in results if item["model"] == "XGBoost")
SEEDS = [1, 2, 3, 4, 5]

def seed_maes(columns, params, frame=train):
    """MAE validation của XGBoost với từng seed."""
    values = []
    for seed in SEEDS:
        model = make_model("XGBoost", columns).set_params(**params, random_state=seed)
        model.fit(frame[columns], frame["traffic_volume"])
        values.append(regression_metrics(validation["traffic_volume"], model.predict(validation[columns]))["MAE"])
    return np.array(values)

maes_a = seed_maes(CONFIG_A, params_a)
maes_b = seed_maes(CONFIG_B, params_b)
# Đánh giá thêm A khi dùng đúng tham số của B để loại khác biệt do tuning.
maes_a_same = seed_maes(CONFIG_A, params_b)
display(pd.DataFrame(
    {"MAE_tb": [m.mean() for m in (maes_a, maes_a_same, maes_b)],
     "do_lech_chuan": [m.std(ddof=1) for m in (maes_a, maes_a_same, maes_b)],
     "nho_nhat": [m.min() for m in (maes_a, maes_a_same, maes_b)],
     "lon_nhat": [m.max() for m in (maes_a, maes_a_same, maes_b)]},
    index=["A, tham số của A", "A, tham số của B", "B, tham số của B"],
).round(2))

,MAE_tb,do_lech_chuan,nho_nhat,lon_nhat
"A, tham số của A",240.14,0.22,239.76,240.29
"A, tham số của B",241.18,0.24,240.99,241.45
"B, tham số của B",238.78,0.49,238.09,239.40


**Nhận xét**

- Thêm thời tiết giảm MAE rất ít: XGBoost từ 244,94 xuống 243,45 (0,6%), Random Forest từ 248,71 xuống 245,83 (1,2%), Ridge từ 283,10 xuống 280,61 (0,9%). Khoảng tin cậy của cả ba đều chứa 0.
- RMSE của Random Forest và XGBoost tăng nhẹ (396,7 → 399,5 và 385,9 → 389,9), nên chưa thể nói B tốt hơn A về sai số lớn.
- Với 5 seed, A đạt MAE 244,37 ± 0,39 và B đạt 242,72 ± 0,61. Chênh lệch 1,65 lớn hơn dao động do seed nhưng chỉ khoảng 0,7%. A dùng tham số của A hay của B cho cùng kết quả vì tuning đã chọn cùng một bộ tham số.
- Kết luận trong thiết kế hiện tại: thời tiết quan trắc cùng giờ chỉ bổ sung rất ít khi đã có thời gian và ngày lễ. Kết quả này không có nghĩa thời tiết không ảnh hưởng đến giao thông; mục 3.5 cho thấy tác động tập trung ở một số điều kiện.

## 3.5 · Thời tiết giúp ở điều kiện nào

So sánh XGBoost của A và B theo nhóm thời tiết. Các nhóm chồng lấp vì một giờ có thể mang nhiều nhãn, nên không cộng các dòng trong bảng. Nhóm có ít giờ cần diễn giải thận trọng.

In [19]:
item_b = next(item for item in results if item["model"] == "XGBoost")
prediction_a = previous["A_lich_le|XGBoost"].to_numpy()
rows = []
for name in WEATHER_GROUPS:
    mask = validation[f"weather_{name}"].eq(1).to_numpy()
    if not mask.any():
        continue
    part = validation[mask]
    rows.append({
        "nhom": name, "so_gio": int(mask.sum()),
        "luu_luong_tb": part["traffic_volume"].mean(),
        "MAE_A": np.abs(prediction_a[mask] - part["traffic_volume"]).mean(),
        "MAE_B": np.abs(item_b["prediction"][mask] - part["traffic_volume"]).mean(),
        "sai_so_tb_B": (item_b["prediction"][mask] - part["traffic_volume"]).mean(),
    })
weather_table = pd.DataFrame(rows).sort_values("so_gio", ascending=False)
weather_table["giam_MAE"] = weather_table["MAE_A"] - weather_table["MAE_B"]
display(weather_table.round(1))

print("Số giờ có snow_1h > 0: validation", int(validation["snow_1h"].gt(0).sum()), "| train", int(train["snow_1h"].gt(0).sum()))

# Tập trung vào giờ ban ngày của ngày làm việc, nơi thời tiết xấu có thể làm giảm lưu lượng rõ nhất.
daytime = validation["is_weekend"].eq(0) & validation["is_holiday"].eq(0) & validation["hour"].between(6, 18)
slices = {
    "Mọi giờ ban ngày làm việc": daytime,
    "Có tuyết": daytime & validation["weather_snow"].eq(1),
    "Tuyết lớn (heavy snow)": daytime & validation["weather_description"].str.contains("heavy snow"),
    "Mưa lớn (heavy rain / very heavy rain)": daytime & validation["weather_description"].str.contains("heavy intensity rain|very heavy rain"),
    "Giông": daytime & validation["weather_thunderstorm"].eq(1),
    "Dưới −15 °C": daytime & validation["temp_c"].lt(-15),
}
rows = []
for name, mask in slices.items():
    mask = mask.to_numpy()
    actual = validation.loc[mask, "traffic_volume"].to_numpy()
    rows.append({
        "nhom": name, "so_gio": int(mask.sum()), "so_ngay": validation.loc[mask, "date_time"].dt.normalize().nunique(),
        "MAE_A": np.abs(prediction_a[mask] - actual).mean(),
        "MAE_B": np.abs(item_b["prediction"][mask] - actual).mean(),
        "sai_so_tb_A": (prediction_a[mask] - actual).mean(),
        "sai_so_tb_B": (item_b["prediction"][mask] - actual).mean(),
    })
display(pd.DataFrame(rows).round(1))

part = validation[daytime.to_numpy()]
print("A → B, giờ ban ngày làm việc:", bootstrap_mae_difference(
    part, prediction_a[daytime.to_numpy()], item_b["prediction"][daytime.to_numpy()]))

,nhom,so_gio,luu_luong_tb,MAE_A,MAE_B,sai_so_tb_B,giam_MAE
1,clouds,2120,3954.3,244.3,237.7,-123.9,6.6
0,clear,2115,2914.6,245.4,243.9,-47.5,1.5
5,mist,905,3011.6,219.5,216.9,-71.6,2.5
6,rain,783,3417.3,214.9,215.5,-55.6,-0.6
2,drizzle,363,3375.1,208.7,204.7,-67.3,4.1
8,snow,362,3259.7,331.8,346.8,-32.6,-15.0
9,thunderstorm,235,3137.9,197.3,210.1,-42.3,-12.9
4,haze,228,3297.8,368.9,364.7,-82.2,4.1
3,fog,162,3097.2,194.6,193.9,-133.8,0.8
7,smoke,1,5952.0,65.8,12.3,12.3,53.6


Số giờ có snow_1h > 0: validation 0 | train 31


,nhom,so_gio,so_ngay,MAE_A,MAE_B,sai_so_tb_A,sai_so_tb_B
0,Mọi giờ ban ngày làm việc,2237,173,277.8,259.5,-35.7,-31.5
1,Có tuyết,160,23,489.9,498.4,235.1,53.2
2,Tuyết lớn (heavy snow),1,1,792.4,160.0,792.4,-160.0
3,Mưa lớn (heavy rain / very heavy rain),27,17,262.4,260.0,222.5,184.0
4,Giông,75,31,244.0,254.8,37.3,-36.1
5,Dưới −15 °C,111,11,435.8,394.9,359.5,276.5


A → B, giờ ban ngày làm việc: {'chenh_lech_MAE': 18.2650379492973, 'CI95_thap': 8.122627311125662, 'CI95_cao': 29.68041396992398, 'so_ngay': 173}


**Nhận xét**

- Trên toàn validation, mức thay đổi theo nhóm thời tiết nhỏ và không cùng chiều: `haze` giảm 11,2, `clouds` và `mist` giảm khoảng 5, còn `snow` và `thunderstorm` tăng 5,7 và 9,7.
- Khi chỉ xét giờ ban ngày (06:00–18:00) của ngày làm việc không lễ, thời tiết giảm MAE từ 276,9 xuống 262,4. Mức giảm 14,47 xe/giờ có CI 95% từ 5,03 đến 25,05 trên 173 ngày, rõ hơn nhiều so với toàn bộ validation.
- Ở các giờ đó, A dự đoán cao hơn thực tế khi có tuyết (sai số trung bình +218,5 trên 160 giờ / 23 ngày) và khi dưới −15 °C (+343,9 trên 111 giờ / 11 ngày). B giảm các sai lệch này còn +34,4 và +243,1. Thời tiết xấu làm lưu lượng ban ngày giảm và B đã học được một phần.
- Nhóm `heavy snow` chỉ có 1 giờ và mưa lớn có 27 giờ, không đủ để kết luận riêng.

### 3.5.1 · Giờ ngày lễ và giờ thường

Cell sau tách MAE của A và B theo giờ ngày lễ, đồng thời thử ba mức `colsample_bytree` để kiểm tra chênh lệch có do việc lấy mẫu cột khi số đặc trưng tăng hay không. Mỗi mức chạy 3 seed; các tham số khác giữ theo V_3.

In [20]:
holiday_mask = validation["is_holiday"].eq(1).to_numpy()
actual = validation["traffic_volume"].to_numpy()
rows = []
for colsample in [0.6, 0.8, 1.0]:
    for label, columns in [("A", CONFIG_A), ("B", CONFIG_B)]:
        values = []
        for seed in SEEDS[:3]:
            model = make_model("XGBoost", columns).set_params(
                **{**params_b, "colsample_bytree": colsample}, random_state=seed
            )
            model.fit(train[columns], train["traffic_volume"])
            errors = np.abs(np.clip(model.predict(validation[columns]), 0, None) - actual)
            values.append((errors.mean(), errors[holiday_mask].mean(), errors[~holiday_mask].mean()))
        rows.append({"colsample_bytree": colsample, "cau_hinh": label,
                     **dict(zip(["MAE_tat_ca", "MAE_gio_le", "MAE_gio_thuong"], np.mean(values, axis=0)))})
display(pd.DataFrame(rows).round(1))
print("Số giờ lễ validation:", int(holiday_mask.sum()), "| train:", int(train["is_holiday"].sum()))

,colsample_bytree,cau_hinh,MAE_tat_ca,MAE_gio_le,MAE_gio_thuong
0,0.6,A,241.7,406.1,234.9
1,0.6,B,239.0,518.7,227.5
2,0.8,A,241.3,397.1,234.9
3,0.8,B,239.0,527.7,227.2
4,1.0,A,241.8,391.4,235.6
5,1.0,B,240.5,525.3,228.8


Số giờ lễ validation: 240 | train: 844


**Nhận xét**

- Ở cả ba mức `colsample_bytree`, B có MAE giờ thường thấp hơn A khoảng 6 xe/giờ (228,6–230,4 so với 235,3–236,6), nhưng MAE giờ lễ cao hơn 108–139 xe/giờ (567,8–575,5 so với 433,9–467,6).
- Chênh lệch ở giờ lễ không mất đi khi lấy toàn bộ cột ở mỗi cây, nên nguyên nhân không chỉ là lấy mẫu cột. Train chỉ có 844 giờ lễ; khi có thêm cột thời tiết, mô hình có thể chia nhỏ các giờ lễ theo thời tiết với rất ít mẫu. Đây là giả thuyết, chưa được kiểm chứng.
- Phần lợi ích ở giờ thường bị bù trừ bởi 240 giờ lễ, đây là một lý do khiến tổng MAE của A và B gần nhau. Hướng thử tiếp: tăng `min_child_weight`, hoặc mô hình riêng/ràng buộc cho ngày lễ, đánh giá bằng CV trong train.

## 3.6 · Độ nhạy với các quyết định dữ liệu còn để ngỏ

`data_validation.ipynb` đã ghi nhận các vấn đề chưa có quy tắc xử lý chính thức. Giữ nguyên tham số đã chọn của XGBoost và chỉ thay đổi dữ liệu train; validation không đổi. Đây là phép kiểm tra độ nhạy, chưa phải đề xuất đổi quy tắc.

| Phương án | Nội dung |
|---|---|
| Che nhiệt độ lặp | Coi `temp = 276.793` là thiếu; giá trị này lặp nhiều giờ liên tiếp vào mùa đông 2013–2014 |
| Bổ sung ngày lễ | Thêm các ngày lễ liên bang thiếu nhãn (xem V_2) |
| Bỏ đợt đóng đường | Bỏ các giờ 22/07/2016 20:00 – 24/07/2016 17:00 khỏi train, theo lịch đóng đường trong `docs/station_data_info.docx` |

In [21]:
stuck = load_splits(mask_stuck_temp=True)
changed_train = stuck["train"]["temp"].to_numpy() != splits["train"]["temp"].to_numpy()
print("Số giờ train đổi temp:", int(changed_train.sum()),
      "| validation đổi:", int((stuck["validation"]["temp"].to_numpy() != splits["validation"]["temp"].to_numpy()).sum()))
print("Khoảng thời gian:", splits["train"].loc[changed_train, "date_time"].min(), "đến", splits["train"].loc[changed_train, "date_time"].max())
print("temp trước khi che (K): trung bình", round(splits["train"].loc[changed_train, "temp"].mean(), 2),
      "| sau khi che:", round(stuck["train"].loc[changed_train, "temp"].mean(), 2))

closure = train["date_time"].between("2016-07-22 20:00", "2016-07-24 17:00")
print("Số giờ đóng đường trong train:", int(closure.sum()))

variants = {
    "Quy tắc hiện tại": train,
    "Che nhiệt độ lặp": build_features(stuck)["train"],
    "Bổ sung ngày lễ": build_features(load_splits(complete_holidays=True))["train"],
    "Bỏ đợt đóng đường": train[~closure],
}
rows = []
for name, frame in variants.items():
    maes = seed_maes(CONFIG_B, params_b, frame)
    rows.append({
        "phuong_an": name, "so_gio_train": len(frame),
        "MAE_tb_5_seed": maes.mean(), "do_lech_chuan": maes.std(ddof=1),
        "nho_nhat": maes.min(), "lon_nhat": maes.max(),
    })
sensitivity = pd.DataFrame(rows)
sensitivity["so_voi_hien_tai"] = sensitivity["MAE_tb_5_seed"].iloc[0] - sensitivity["MAE_tb_5_seed"]
display(sensitivity.round(2))

Số giờ train đổi temp: 78 | validation đổi: 0
Khoảng thời gian: 2013-10-22 05:00:00 đến 2014-01-23 08:00:00
temp trước khi che (K): trung bình 276.34 | sau khi che: 261.94
Số giờ đóng đường trong train: 46


,phuong_an,so_gio_train,MAE_tb_5_seed,do_lech_chuan,nho_nhat,lon_nhat,so_voi_hien_tai
0,Quy tắc hiện tại,28402,238.78,0.49,238.09,239.40,0.00
1,Che nhiệt độ lặp,28402,238.84,0.71,238.13,239.98,-0.06
2,Bổ sung ngày lễ,28402,237.92,0.80,237.11,239.05,0.86
3,Bỏ đợt đóng đường,28356,234.99,0.68,234.01,235.74,3.79


**Nhận xét**

- Che giá trị lặp `276.793` đổi 78 giờ train từ 22/10/2013 đến 23/01/2014; nhiệt độ trung bình của các giờ này giảm từ 276,34 K xuống 261,94 K, phù hợp hơn với mùa đông. MAE thay đổi −0,40, nằm trong dao động do seed (độ lệch chuẩn khoảng 0,4–0,7). Quyết định nên dựa vào tính hợp lệ của dữ liệu, không dựa vào MAE.
- Bổ sung ngày lễ giảm 1,16, khoảng 1,6–1,9 lần độ lệch chuẩn theo seed: tín hiệu yếu.
- Bỏ 46 giờ đóng đường giảm MAE trung bình 4,11 (238,61 so với 242,72), lớn hơn nhiều so với dao động do seed. Các giờ này không phản ánh hành vi giao thông thông thường nên có cơ sở để loại khỏi train.
- Cả ba phương án được chọn ra và đo trên cùng validation. Nếu nhóm áp dụng, cần ghi lý do theo dữ liệu, ghi vào quy tắc tiền xử lý và chạy lại toàn bộ phiên bản để giữ so sánh công bằng.

## 3.7 · Ghi nhật ký thực nghiệm

In [22]:
# Log chỉ được tạo khi chủ động bật; đặt biến môi trường SAVE_EXPERIMENT_LOGS=1
# hoặc đổi giá trị bên dưới thành True. Mỗi lần ghi tạo file mới, không ghi đè.
SAVE_EXPERIMENT_LOGS = os.environ.get("SAVE_EXPERIMENT_LOGS") == "1"

common_observations = run_observations(data_path, splits, "notebooks/experiments/V_3.ipynb")
common_problems = ['Dùng thời tiết quan trắc cùng giờ; chưa đo khi chỉ có dự báo thời tiết.', 'Dữ liệu thiếu giờ; validation chỉ gồm các giờ quan sát được.']
next_steps = ['Đánh giá thêm lịch sử lưu lượng trong V_4 và phân tích sai số.']

if SAVE_EXPERIMENT_LOGS:
    for item in results:
        log_path = save_experiment_log(
            model=f"{item['config']} - {item['model']}",
            features=item["features"],
            results=item["metrics"],
            observations=common_observations + [
                "Tập đánh giá: " + item.get("tap_danh_gia", f"validation; {len(validation)} giờ."),
                f"Tham số được chọn: {item['search'].best_params_}",
                f"MAE trung bình CV: {-item['search'].best_score_}",
                f"Kết quả từng fold: {summarize_search(item['search']).to_json(orient='records', force_ascii=False)}",
            ],
            problems=common_problems,
            next_steps=next_steps,
        )
        print("Đã lưu:", log_path.name)
else:
    print("Chưa tạo log. Đặt SAVE_EXPERIMENT_LOGS=1 khi cần ghi.")

Chưa tạo log. Đặt SAVE_EXPERIMENT_LOGS=1 khi cần ghi.


## 3.8 · Kết luận

| Mô hình | MAE A | MAE B | Giảm | CI 95% |
|---|---|---|---|---|
| XGBoost | 244,94 | **243,45** | 1,49 | −5,14 đến 7,66 |
| Random Forest | 248,71 | 245,83 | 2,88 | −3,35 đến 8,57 |
| Ridge | 283,10 | 280,61 | 2,49 | −0,50 đến 5,59 |

- Trả lời câu hỏi nghiên cứu 2 trong thiết kế hiện tại: thời tiết cải thiện MAE khoảng 0,6–1,2% trên toàn validation, chưa đủ rõ so với dao động theo ngày. Lợi ích rõ hơn ở giờ ban ngày ngày làm việc (giảm 14,5 xe/giờ), nhưng bị bù trừ ở giờ lễ.
- Demo Time + Weather nên trình bày đúng mức đóng góp này và không mô tả thời tiết là yếu tố chính.
- Còn mở: thời tiết dự báo thay cho quan trắc; quy tắc nhiệt độ lặp và đợt đóng đường cần nhóm chốt.